### Basic Setups

In [1]:
import os
import cv2
import matplotlib.pyplot as plt
import random
from ultralytics import YOLO
import shutil
from pathlib import Path
import yaml

### Data Preprocessing

In [2]:
RawDataset = Path("Dataset")      
MasterDatasetFolder = Path("Master Dataset (YOLO)")

for split in ['train', 'val', 'test']:
    (MasterDatasetFolder / split / 'images').mkdir(parents=True, exist_ok=True)
    (MasterDatasetFolder / split / 'labels').mkdir(parents=True, exist_ok=True)

TestLocations = ["Location A"]
TrainValLocations = ["Location B", "Location C", "Location D", "Location E", "Location F"]

def Create():
    random.seed(42)
    
    MyClassNames = ['car', 'heavy_vehicle', 'motorcycle'] 
    
    total_train = total_val = total_test = 0
    
    AllLocations = TestLocations + TrainValLocations
    
    for FolderName in AllLocations:
        print(f"\n--- Processing & Copying: {FolderName} ---")
        LocationPath = RawDataset / FolderName
        
        ImageFolder = LocationPath / "images"
        LabelFolder = LocationPath / "labels"
        
        if not ImageFolder.exists():
            print(f"  [ERROR] Could not find 'images' folder in {FolderName}. Skipping.")
            continue
            
        images = []
        for ext in ('*.jpg', '*.jpeg', '*.png'):
            images.extend(list(ImageFolder.glob(ext)))
            
        if not images:
            print(f"  [WARNING] No images found in {ImageFolder}.")
            continue
            
        if FolderName in TestLocations:
            test_images = images
            train_images = []
            val_images = []
            print(f"  Allocating {len(test_images)} images to TEST.")
        else:
            random.shuffle(images)
            split_index = int(len(images) * 0.8)
            train_images = images[:split_index]
            val_images = images[split_index:]
            test_images = []
            print(f"  Allocating {len(images)} images -> Train: {len(train_images)} | Val: {len(val_images)}")

        def copy_files(file_list, split_name):
            copied_count = 0
            for img_path in file_list:
                dest_img = MasterDatasetFolder / split_name / "images" / img_path.name
                shutil.copy(img_path, dest_img)
                
                label_name = img_path.stem + ".txt"
                source_label = LabelFolder / label_name
                
                if source_label.exists():
                    dest_label = MasterDatasetFolder / split_name / "labels" / label_name
                    shutil.copy(source_label, dest_label)
                
                copied_count += 1
            return copied_count

        total_train += copy_files(train_images, 'train')
        total_val += copy_files(val_images, 'val')
        total_test += copy_files(test_images, 'test')

    if total_train == 0 and total_val == 0 and total_test == 0:
        print("\n❌ [CRITICAL ERROR] No files were copied. Verify your directory paths.")
        return

    MasterYaml = {
        'path': str(MasterDatasetFolder.resolve()),
        'train': 'train/images',
        'val': 'val/images',
        'test': 'test/images',
        'names': MyClassNames
    }
    
    with open(MasterDatasetFolder / 'data.yaml', 'w') as f: 
        yaml.dump(MasterYaml, f, sort_keys=False)
        
    print(f"\n✅ Physical dataset and master data.yaml successfully built inside '{MasterDatasetFolder.name}'!")
    print(f"  Total Train Images Copied: {total_train}")
    print(f"  Total Val Images Copied:   {total_val}")
    print(f"  Total Test Images Copied:  {total_test}")

Create()


--- Processing & Copying: Location A ---
  Allocating 182 images to TEST.

--- Processing & Copying: Location B ---
  Allocating 966 images -> Train: 772 | Val: 194

--- Processing & Copying: Location C ---
  Allocating 1092 images -> Train: 873 | Val: 219

--- Processing & Copying: Location D ---
  Allocating 838 images -> Train: 670 | Val: 168

--- Processing & Copying: Location E ---
  Allocating 200 images -> Train: 160 | Val: 40

--- Processing & Copying: Location F ---
  Allocating 391 images -> Train: 312 | Val: 79

✅ Physical dataset and master data.yaml successfully built inside 'Master Dataset (YOLO)'!
  Total Train Images Copied: 2787
  Total Val Images Copied:   700
  Total Test Images Copied:  182


### Model Preparation

In [3]:
model = YOLO('Transfer Learning Model Path/yolov8n.pt')

model.info(detailed=True)

layer                                    name                type  gradient  parameters               shape        mu     sigma
    0                     model.0.conv.weight              Conv2d     False         432       [16, 3, 3, 3]  -0.00279     0.152        float32
    1                       model.0.bn.weight         BatchNorm2d     False          16                [16]      2.97      1.86        float32
    1                         model.0.bn.bias         BatchNorm2d     False          16                [16]     0.249      4.17        float32
    2                             model.0.act                SiLU     False           0                  []         -         -              -
    3                     model.1.conv.weight              Conv2d     False        4608      [32, 16, 3, 3]  -0.00012     0.063        float32
    4                       model.1.bn.weight         BatchNorm2d     False          32                [32]      5.02      1.12        float32
    4         

(129, 3157200, 0, 8.8575488)

### Phase I - Warmup

In [4]:
DataPath = 'Master Dataset (YOLO)/data.yaml'

# ==========================================
# STAGE 1 : WARMUP (Frozen Backbone)
# ==========================================
model = YOLO('Transfer Learning Model Path/yolov8n.pt')

model.train(
    data         = DataPath, 
    epochs       = 15,          
    freeze       = 22,                  # Freeze Layer 0 - 21
    lr0          = 0.01,
    project      = 'Transfer Learning',
    name         = 'Warmup',
    device       = 0,
    imgsz        = 640,
    amp          = True,
    batch        = 8,  
    workers      = 2,
    cache        = False,      
    patience     = 10,       
    exist_ok     = True,
    
    # --- Augmentations ---
    mosaic       = 1.0,                 # Help fight scale and small boxes
    scale        = 0.5,                 # Help fight spatial bias
    fliplr       = 0.5,                 # Help fight horizontal spatial bias
    translate    = 0.1,                 # Help fight pixel spot bias
    hsv_v        = 0.1,                 # Help fight weather oclusions through brightness jitter
    hsv_s        = 0.1,                 # Help fight weather oclusions through saturation jitter
    hsv_h        = 0.1                  # Help fight weather oclusions through hue jitter
)

New https://pypi.org/project/ultralytics/8.4.60 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.222  Python-3.10.18 torch-2.5.1 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=Master Dataset (YOLO)/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=15, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=22, half=False, hsv_h=0.1, hsv_s=0.1, hsv_v=0.1, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=Transfer Learning Model Path/yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, na

c:\Users\justi\anaconda3\envs\pytorch\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access  (ping: 0.10.0 ms, read: 7.91.3 MB/s, size: 49.8 KB)
val: Scanning C:\Users\justi\Documents\Project\SoCS\Python\Computer Vision AOL\Master Dataset (YOLO)\val\labels... 700 images, 2 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 700/700 476.4it/s 1.5s0.0s
val: New cache created: C:\Users\justi\Documents\Project\SoCS\Python\Computer Vision AOL\Master Dataset (YOLO)\val\labels.cache
Plotting labels to C:\Users\justi\Documents\Project\SoCS\Python\Computer Vision AOL\Transfer Learning\Warmup\labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.001429, momentum=0.9) with parameter groups 57 weight(decay=0.0), 64 weight(decay=0.0005)

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000001A1EAE9FA00>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          

### Phase II - Fine Tune

In [9]:
model = YOLO('Transfer Learning/Warmup/weights/best.pt') 

model.train(
    data         = DataPath, 
    epochs       = 100,          
    freeze       = 0,                   # Unfreeze all
    lr0          = 0.001,               # Reduce lr 10x to avoid explosion
    cos_lr       = True,      
    project      = 'Transfer Learning',
    name         = 'Fine Tune',
    device       = 0,
    imgsz        = 640,        
    amp          = True,
    batch        = 8,
    workers      = 2,
    cache        = False,      
    patience     = 15,
    exist_ok     = True,
    
    # --- Excellent Augmentations ---
    cls          = 5.0,                 # Help fight class imbalance
    box          = 10.0,                # Help fight scale variance
    mosaic       = 1.0,
    close_mosaic = 20,                  
    scale        = 0.8,                 # Wider range for resizing crops
    mixup        = 0.15,                # Help fight class/location imbalance
    copy_paste   = 0.3,                 # Help fight pixel bias
    fliplr       = 0.5,
    translate    = 0.15,                # help fight spatial bias
    perspective  = 0.0005,              # Introduces slight 3D angle distortion
    hsv_v        = 0.4,
    hsv_s        = 0.5 
)

New https://pypi.org/project/ultralytics/8.4.60 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.222  Python-3.10.18 torch-2.5.1 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=10.0, cache=False, cfg=None, classes=None, close_mosaic=20, cls=5.0, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=Master Dataset (YOLO)/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.5, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=Transfer Learning/Warmup/weights/best.pt, momentum=0.937, mosaic=1.0, multi_scale=Fals

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000001A157C7EB60>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          

### Testing

In [11]:
ModelTest = YOLO('Transfer Learning/Fine Tune/weights/best.pt')

TestImageFolder = 'Master Dataset (YOLO)/test/images'
TestImages = [os.path.join(TestImageFolder, f) for f in os.listdir(TestImageFolder) if f.endswith(('.jpg', '.png', '.jpeg'))]

SampleImages = random.sample(TestImages, 6)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, ImagePath in enumerate(SampleImages):
    results = ModelTest.predict(source=ImagePath, conf=0.2, save=False)[0]
    
    res_plotted = results.plot()
    
    res_rgb = cv2.cvtColor(res_plotted, cv2.COLOR_BGR2RGB)
    
    axes[i].imshow(res_rgb)
    axes[i].set_title(f"Test Image {i+1}")
    axes[i].axis('off')

plt.tight_layout()
plt.show()

ModelTest.predict(source=TestImageFolder, conf=0.2, save=True, project='Transfer Learning', name='Fine Tune/Test Results')
print(f"All annotated images saved to: Transfer Learning/Fine Tune/Test Results")


image 1/1 c:\Users\justi\Documents\Project\SoCS\Python\Computer Vision AOL\Master Dataset (YOLO)\test\images\video5_frame_12_jpg.rf.1e3bb5c1c4317f4d12add3ed7aef0563.jpg: 384x640 9 cars, 19 motorcycles, 20.0ms
Speed: 2.3ms preprocess, 20.0ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)

image 1/1 c:\Users\justi\Documents\Project\SoCS\Python\Computer Vision AOL\Master Dataset (YOLO)\test\images\video15_frame_6_jpg.rf.c3c56b30c5ca37ac327622884dbf4d2c.jpg: 384x640 6 cars, 2 heavy_vehicles, 10 motorcycles, 17.6ms
Speed: 2.5ms preprocess, 17.6ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

image 1/1 c:\Users\justi\Documents\Project\SoCS\Python\Computer Vision AOL\Master Dataset (YOLO)\test\images\video2_frame_19_jpg.rf.28e538bc30a0befa0c59e02082c84d03.jpg: 384x640 1 car, 13 motorcycles, 16.9ms
Speed: 1.9ms preprocess, 16.9ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

image 1/1 c:\Users\justi\Documents\Project\SoCS\Python\Comput

<Figure size 1800x1000 with 6 Axes>


image 1/182 c:\Users\justi\Documents\Project\SoCS\Python\Computer Vision AOL\Master Dataset (YOLO)\test\images\frame0_jpg.rf.7f2e207aa9cfdbccdbf8284fe45a7779.jpg: 384x640 11 cars, 5 heavy_vehicles, 24 motorcycles, 12.5ms
image 2/182 c:\Users\justi\Documents\Project\SoCS\Python\Computer Vision AOL\Master Dataset (YOLO)\test\images\frame10_jpg.rf.eccda67e0bcead9a6d1b284485481be4.jpg: 384x640 7 cars, 6 heavy_vehicles, 16 motorcycles, 12.6ms
image 3/182 c:\Users\justi\Documents\Project\SoCS\Python\Computer Vision AOL\Master Dataset (YOLO)\test\images\frame12_jpg.rf.40a342bdff710c2d6f08b0603fbcd989.jpg: 384x640 13 cars, 26 motorcycles, 13.9ms
image 4/182 c:\Users\justi\Documents\Project\SoCS\Python\Computer Vision AOL\Master Dataset (YOLO)\test\images\frame13_jpg.rf.423ef1573f3f2271ceb892c7d8a7ee58.jpg: 384x640 19 cars, 1 heavy_vehicle, 25 motorcycles, 12.4ms
image 5/182 c:\Users\justi\Documents\Project\SoCS\Python\Computer Vision AOL\Master Dataset (YOLO)\test\images\frame14_jpg.rf.e1a471